In [ ]:
import pandas as pd

# ── Helper: print a section header ───────────────────────────
def section(title):
    print(f"\n{'─' * 50}")
    print(f"  {title}")
    print(f"{'─' * 50}")

In [10]:
# ============================================================
# STEP 1 – Load the SQL-extracted file
# ============================================================

section("STEP 1 - Load data")

df = pd.read_csv("visits_clean.csv")

print(f"  Rows    : {len(df)}")
print(f"  Columns : {df.shape[1]}")
print(f"  Columns : {df.columns.tolist()}")


──────────────────────────────────────────────────
  STEP 1 - Load data
──────────────────────────────────────────────────
  Rows    : 897
  Columns : 22
  Columns : ['patient_id', 'patient_name', 'gender', 'age', 'visit_date', 'address', 'consulted_by', 'treatment', 'dental_camp', 'xray_done', 'patient_type', 'treatment_type', 'treatment_cost', 'xray_cost', 'opd_cost', 'lab_charges', 'discount', 'total_paid', 'due', 'profit', 'payment_method', 'visit_month']


In [11]:
# ============================================================
# STEP 2 – Data quality check
# ============================================================
# Always verify the data before touching it.
# We check for missing values and duplicate rows.

section("STEP 2 · Data quality check")

missing = df.isnull().sum()
print("\n  Missing values per column:")
if missing.any():
    print(missing[missing > 0].to_string())
else:
    print("  ✔ No missing values")

dupes = df.duplicated().sum()
print(f"\n  Duplicate rows : {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"  ✔ Removed. Rows remaining: {len(df)}")
else:
    print("  ✔ No duplicates")



──────────────────────────────────────────────────
  STEP 2 · Data quality check
──────────────────────────────────────────────────

  Missing values per column:
  ✔ No missing values

  Duplicate rows : 0
  ✔ No duplicates


In [12]:
# ============================================================
# STEP 3 – Fix visit_date data type
# ============================================================
# SQL exported visit_date as plain text ("02-01-2023").
# We convert it to a proper date so Tableau can use it
# for time-based filtering and sorting correctly.

section("STEP 3 · Fix visit_date data type")

df["visit_date"] = pd.to_datetime(df["visit_date"], format="%d-%m-%Y")

print(f"  ✔ visit_date converted from text → datetime")
print(f"  Sample : {df['visit_date'].iloc[0].date()}")
print(f"  Range  : {df['visit_date'].min().date()}  →  {df['visit_date'].max().date()}")


──────────────────────────────────────────────────
  STEP 3 · Fix visit_date data type
──────────────────────────────────────────────────
  ✔ visit_date converted from text → datetime
  Sample : 2023-01-02
  Range  : 2023-01-02  →  2024-12-07


In [13]:
# ============================================================
# STEP 4 – Fix numeric data types
# ============================================================
# These columns came out of SQL as float (e.g. 411.0, 350.0).
# Most are whole numbers — we convert them to integer.
# NOTE: discount, total_paid, and profit have a few genuine
# decimal values (e.g. 397.2), so we leave those as floatsection("STEP 4 · Fix numeric data types")

whole_number_cols = ["treatment_cost", "xray_cost", "opd_cost", "lab_charges", "due"]

df[whole_number_cols] = df[whole_number_cols].astype(int)

print(f"  ✔ Converted to int : {whole_number_cols}")
print(f"  ✔ Kept as float    : ['discount', 'total_paid', 'profit']")
print(f"     (these contain genuine decimal values from partial discounts)")




  ✔ Converted to int : ['treatment_cost', 'xray_cost', 'opd_cost', 'lab_charges', 'due']
  ✔ Kept as float    : ['discount', 'total_paid', 'profit']
     (these contain genuine decimal values from partial discounts)


In [14]:
# ============================================================
# STEP 5 – Add derived columns for Tableau
# ============================================================
# We create three new columns that make Tableau filters easier.
# This is better than calculating them inside Tableau itself.
section("STEP 5 · Add derived columns")

# Year and month number — useful for Tableau date filters
df["visit_year"]      = df["visit_date"].dt.year
df["visit_month_num"] = df["visit_date"].dt.month

# Age group — groups raw age into readable bands
df["age_group"] = pd.cut(
    df["age"],
    bins   = [0,  17,  35,  50,  65, 100],
    labels = ["Under 18", "18–35", "36–50", "51–65", "65+"]
)

print("  ✔ Added: visit_year, visit_month_num")
print("  ✔ Added: age_group")
print(f"\n  Age group breakdown:\n{df['age_group'].value_counts().sort_index().to_string()}")


──────────────────────────────────────────────────
  STEP 5 · Add derived columns
──────────────────────────────────────────────────
  ✔ Added: visit_year, visit_month_num
  ✔ Added: age_group

  Age group breakdown:
age_group
Under 18    172
18–35       187
36–50       167
51–65       206
65+         165


In [16]:
# ============================================================
# STEP 6 – Final validation
# ============================================================
# A last check to confirm the data is clean and nothing broke.

section("STEP 6 - Final validation")

print(f"\n  Rows          : {len(df)}")
print(f"  Columns       : {df.shape[1]}")
print(f"  Date range    : {df['visit_date'].min().date()}  →  {df['visit_date'].max().date()}")
print(f"  Nulls left    : {df.isnull().any().any()}")
print(f"\n  Final data types:\n{df.dtypes.to_string()}")



──────────────────────────────────────────────────
  STEP 6 - Final validation
──────────────────────────────────────────────────

  Rows          : 897
  Columns       : 25
  Date range    : 2023-01-02  →  2024-12-07
  Nulls left    : False

  Final data types:
patient_id                 object
patient_name               object
gender                     object
age                         int64
visit_date         datetime64[ns]
address                    object
consulted_by               object
treatment                  object
dental_camp                object
xray_done                  object
patient_type               object
treatment_type             object
treatment_cost              int64
xray_cost                   int64
opd_cost                    int64
lab_charges                 int64
discount                  float64
total_paid                float64
due                         int64
profit                    float64
payment_method             object
visit_month           

In [17]:
# ============================================================
# STEP 7 – Export
# ============================================================

section("STEP 7 · Export")

output_file = "visits_ready.csv"
df.to_csv(output_file, index=False)

print(f"\n  ✔ Clean file saved : {output_file}")
print("  → Ready to import into Tableau\n")


──────────────────────────────────────────────────
  STEP 7 · Export
──────────────────────────────────────────────────

  ✔ Clean file saved : visits_ready.csv
  → Ready to import into Tableau

